# Pratica 04 — Rappresentazioni sparse

**Corso:** NLP Classico con Python · **Lezione 04** · **Tempo stimato:** 60-75 minuti
**Notebook di lezione:** `notebooks_clase/clase04_representaciones_dispersas_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- I testi degli esercizi da 1 a 5 sono nella cella di dati che segue il setup; la sfida usa le recensioni di `datos/` (in Colab, carica la cartella `datos/`). Sono testi sintetici scritti per il corso (*esempio illustrativo*).
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Librerie: `numpy`, `pandas`, `scipy` e `scikit-learn`.

---
## Cella 0 — Setup e dati

In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

ARCHIVO_RESENAS = {"spa": "resenas_es.csv", "ita": "resenas_it.csv"}
print("Librerie caricate correttamente.")

In [ ]:
# datos-es
FRASES = {
    "spa": ["la entrega fue rápida",
           "la entrega fue lenta",
           "el servicio fue rápido y amable",
           "la atención fue amable"],
    "ita": ["la consegna è stata veloce",
           "la consegna è stata lenta",
           "il servizio è stato veloce e cortese",
           "il personale è stato cortese"],
}
FRASE_NEGACION = {"spa": "no es bueno, no es malo", "ita": "non è buono, non è cattivo"}

## Esercizio 1 — Sacco di parole a mano

Completa `bolsa(frases)`: deve restituire `(vocabulario, matriz)`, dove `vocabulario` è la lista **ordinata** delle parole distinte (tokenizzate con `\w+` e in minuscolo) e `matriz[i][j]` è il numero di volte che `vocabulario[j]` compare nella frase `i` (lista di liste di interi). Si verifica con `CountVectorizer` nelle due lingue.

Prima di programmare, conta a mano quante parole distinte ha `FRASES["spa"]`.

In [ ]:
def bolsa(frases):
    tokens = [re.findall(r"\w+", f.lower()) for f in frases]
    vocabulario = None      # TODO: lista ordinata di parole distinte
    matriz = None           # TODO: lista di liste di conteggi
    return vocabulario, matriz

vocab_es, matriz_es = bolsa(FRASES["spa"])
vocab_it, matriz_it = bolsa(FRASES["ita"])

# Controllo automatico
assert vocab_es is not None and matriz_es is not None and vocab_it is not None and matriz_it is not None, "Completa le righe contrassegnate con TODO"
assert len(vocab_es) == 11 and len(vocab_it) == 12, "il vocabolario ha 11 parole in spagnolo e 12 in italiano"
assert [sum(f) for f in matriz_es] == [4, 4, 6, 4], "la somma di ogni riga è il numero di parole della frase"
assert [sum(f) for f in matriz_it] == [5, 5, 7, 5], "la somma di ogni riga è il numero di parole della frase"
for idioma, vocab, matriz in (("spa", vocab_es, matriz_es), ("ita", vocab_it, matriz_it)):
    cv = CountVectorizer(token_pattern=r"\w+")
    esperada = cv.fit_transform(FRASES[idioma]).toarray()
    assert vocab == list(cv.get_feature_names_out()), "il vocabolario deve essere ordinato e senza ripetizioni"
    assert (np.array(matriz) == esperada).all(), "la matrice non coincide con quella di CountVectorizer"
print("Esercizio 1 corretto ✔")

## Esercizio 2 — idf e tf-idf a mano

Su `FRASES["spa"]` (`n = 4` frasi) e con la formula di scikit-learn `idf = ln((1 + n) / (1 + df)) + 1`:

- completa la funzione `idf(n, df)`,
- calcola `idf_entrega` (la parola `entrega`),
- calcola il tf-idf **non normalizzato** di ogni parola della frase 0 (`pesos_brutos`: dizionario parola → `tf · idf`),
- calcola `peso_entrega`: il peso di `entrega` dopo aver diviso per la norma L2 della riga.

Verifica prima a mano quante frasi contengono `entrega`.

In [ ]:
def idf(n, df):
    return None      # TODO: formula di scikit-learn

tokens = [re.findall(r"\w+", f.lower()) for f in FRASES["spa"]]
n = len(tokens)
df_palabra = Counter(p for fila in tokens for p in set(fila))       # frasi che contengono ogni parola

idf_entrega = None       # TODO
pesos_brutos = None      # TODO: {parola: tf · idf} per la frase 0
peso_entrega = None      # TODO: pesos_brutos["entrega"] / norma L2 della riga

# Controllo automatico
assert idf_entrega is not None and pesos_brutos is not None and peso_entrega is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(idf(4, 4), 1.0), "una parola presente in tutte le frasi ha idf = 1"
tv = TfidfVectorizer(token_pattern=r"\w+").fit(FRASES["spa"])
assert np.isclose(idf_entrega, tv.idf_[tv.vocabulary_["entrega"]]), "rivedi la formula: ln((1 + n) / (1 + df)) + 1"
assert np.isclose(idf_entrega, np.log(5 / 3) + 1), "entrega compare in 2 delle 4 frasi: ln(5/3) + 1"
bruta = TfidfVectorizer(token_pattern=r"\w+", norm=None).fit(FRASES["spa"])
fila0 = bruta.transform(FRASES["spa"][:1]).toarray()[0]
assert set(pesos_brutos) == set(tokens[0]), "pesos_brutos deve avere una voce per ogni parola distinta della frase 0"
for palabra, peso in pesos_brutos.items():
    assert np.isclose(peso, fila0[bruta.vocabulary_[palabra]]), f"peso errato per {palabra}"
assert np.isclose(peso_entrega, tv.transform(FRASES["spa"][:1])[0, tv.vocabulary_["entrega"]]), "il peso normalizzato è il valore grezzo diviso per la norma L2 della riga"
print(f"Esercizio 2 corretto ✔ (idf di entrega = {idf_entrega:.4f}, peso normalizzato = {peso_entrega:.4f})")

## Esercizio 3 — Contare i bigrammi

`FRASE_NEGACION` ha una frase per lingua con un bigramma ripetuto. Completa `bigramas(frase)` (lista di coppie di parole consecutive, con `\w+` e in minuscolo) e calcola:

- `n_total`: quanti bigrammi ci sono (con ripetizioni) nella frase in spagnolo,
- `n_distintos`: quanti sono distinti,
- `n_rasgos_1_2`: quante caratteristiche avrebbe un vettorizzatore con `ngram_range=(1, 2)` addestrato solo con quella frase (unigrammi distinti + bigrammi distinti).

Scrivi prima i bigrammi della frase su carta.

In [ ]:
def bigramas(frase):
    tokens = re.findall(r"\w+", frase.lower())
    return None      # TODO: lista di tuple (parola_i, parola_i+1)

n_total = None           # TODO
n_distintos = None       # TODO
n_rasgos_1_2 = None      # TODO

# Controllo automatico
assert n_total is not None and n_distintos is not None and n_rasgos_1_2 is not None and bigramas("a b") is not None, "Completa le righe contrassegnate con TODO"
assert bigramas("uno dos tres") == [("uno", "dos"), ("dos", "tres")], "un bigramma è una coppia di parole consecutive"
assert n_total == 5 and n_distintos == 4, "la frase ha 5 bigrammi e il primo si ripete"
assert n_rasgos_1_2 == 8, "4 parole distinte + 4 bigrammi distinti"
for idioma, frase in FRASE_NEGACION.items():
    cv_bi = CountVectorizer(token_pattern=r"\w+", ngram_range=(2, 2)).fit([frase])
    assert len(cv_bi.vocabulary_) == len(set(bigramas(frase))) == 4
    cv_12 = CountVectorizer(token_pattern=r"\w+", ngram_range=(1, 2)).fit([frase])
    assert len(cv_12.vocabulary_) == 8
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Similarità del coseno a mano

Hai tre vettori di conteggi `u`, `v` e `w` (con `w = 2u`). Implementa `coseno(a, b)` con `numpy` e calcola `cos_uv`, `cos_uw` e `cos_vw`. Calcola prima a mano `cos(u, v)` = `(u · v) / (‖u‖ ‖v‖)` con `u = [2, 1, 0, 1]` e `v = [1, 1, 1, 0]`.

Domanda: perché `cos_uw` vale esattamente 1 anche se `w` ha il doppio dei conteggi?

In [ ]:
u = np.array([2, 1, 0, 1])
v = np.array([1, 1, 1, 0])
w = 2 * u

def coseno(a, b):
    return None      # TODO: prodotto scalare diviso il prodotto delle norme

cos_uv = None        # TODO
cos_uw = None        # TODO
cos_vw = None        # TODO

# Controllo automatico
assert cos_uv is not None and cos_uw is not None and cos_vw is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(cos_uv, 3 / np.sqrt(18)), "u·v = 3, ‖u‖ = √6, ‖v‖ = √3"
assert np.isclose(cos_uw, 1.0), "w ha la stessa direzione di u"
assert np.isclose(cos_vw, cos_uv), "v si confronta con la stessa direzione, quindi cos(v, w) = cos(v, u)"
esperada = cosine_similarity(np.array([u, v, w]))
assert np.allclose([cos_uv, cos_uw, cos_vw], [esperada[0, 1], esperada[0, 2], esperada[1, 2]]), "deve coincidere con cosine_similarity di scikit-learn"
assert cos_uw > cos_uv, "u è più vicino a w che a v"
print(f"Esercizio 4 corretto ✔ (cos(u, v) = {cos_uv:.4f})")

## Esercizio 5 — Prevedere la dimensione del vocabolario con `min_df` e `max_df`

Con le quattro frasi di ciascuna lingua (`n = 4`) calcola **senza usare `CountVectorizer`** quante parole avrebbe il vocabolario con:

- `base`: senza restrizioni,
- `min2`: `min_df=2` (parole in almeno 2 frasi),
- `max075`: `max_df=0.75` (parole in al massimo il 75 % delle frasi, cioè in 3 su 4),
- `ambos`: entrambe le condizioni insieme.

Completa `df_por_palabra` e `predecir`, che restituisce la tupla `(base, min2, max075, ambos)`. Poi si confronta con `CountVectorizer`.

In [ ]:
def df_por_palabra(frases):
    return None      # TODO: Counter parola → numero di frasi che la contengono

def predecir(frases):
    n = len(frases)
    df = df_por_palabra(frases)
    base = None      # TODO
    min2 = None      # TODO
    max075 = None    # TODO
    ambos = None     # TODO
    return base, min2, max075, ambos

previsto = {idioma: predecir(FRASES[idioma]) for idioma in ("spa", "ita")}

# Controllo automatico
assert df_por_palabra(["a b", "a c"]) is not None and all(v is not None for t in previsto.values() for v in t), "Completa le righe contrassegnate con TODO"
assert previsto["spa"] == (11, 4, 10, 3), "spagnolo: 11 parole; con min_df=2 restano la, fue, entrega e amable; con max_df=0.75 se ne va fue (è nelle 4 frasi)"
assert previsto["ita"] == (12, 8, 11, 7), "italiano: 12 parole; con max_df=0.75 se ne va è (è nelle 4 frasi)"
for idioma in ("spa", "ita"):
    real = tuple(len(CountVectorizer(token_pattern=r"\w+", **p).fit(FRASES[idioma]).vocabulary_)
                 for p in ({}, {"min_df": 2}, {"max_df": 0.75}, {"min_df": 2, "max_df": 0.75}))
    assert real == previsto[idioma], f"{idioma}: la tua previsione {previsto[idioma]} non coincide con CountVectorizer {real}"
print("Esercizio 5 corretto ✔")

---
## Sfida finale — Un motore di «recensione più simile»

Costruisci un piccolo motore di similarità sulle **recensioni pulite** (`texto_limpio`) di `datos/`. Scegli la lingua con `IDIOMA` (`"spa"` o `"ita"`).

1. Crea `vectorizador` (un `TfidfVectorizer` con `token_pattern=r"\w+"`) e adattalo a `df_res["texto_limpio"]`; salva la matrice in `X` (deve essere una matrice **sparsa**).
2. Scrivi `mas_similares(i, k=3)`: gli indici delle `k` recensioni più simili alla recensione `i` (per coseno), **senza includere la stessa `i`**, dalla più alla meno simile.
3. Scrivi `buscar(consulta, k=3)`: trasforma un testo nuovo con il vettorizzatore già adattato (`transform`, non `fit`) e restituisce gli indici delle `k` recensioni più simili.

> **Nota:** con 300 recensioni sintetiche il motore trova recensioni con **parole** simili, non necessariamente con lo stesso significato, e le cifre non si generalizzano a collezioni grandi. Serve per esercitare la logica della rappresentazione sparsa.

In [ ]:
IDIOMA = "spa"            # o "ita"
df_res = pd.read_csv(cargar_datos(ARCHIVO_RESENAS[IDIOMA]))

vectorizador = None      # TODO: TfidfVectorizer(token_pattern=r"\w+")
X = None                 # TODO: adatta e trasforma df_res["texto_limpio"]

def mas_similares(i, k=3):
    # TODO: coseno della riga i con tutte le righe; escludi i; ordina dalla più alla meno simile
    return None

def buscar(consulta, k=3):
    # TODO: trasforma la consulta con il vettorizzatore già adattato
    return None

# Controllo automatico
assert vectorizador is not None and X is not None, "Completa le righe contrassegnate con TODO"
assert sp.issparse(X) and X.shape[0] == len(df_res), "X deve essere una matrice sparsa con una riga per recensione"
assert X.nnz / (X.shape[0] * X.shape[1]) < 0.05, "la matrice dovrebbe avere meno del 5 % di celle non nulle"
assert np.allclose(np.sqrt(X.multiply(X).sum(axis=1)).A1, 1.0), "le righe di TfidfVectorizer hanno norma L2 uguale a 1"
for i in range(0, len(df_res), 25):
    vecinos = mas_similares(i, k=3)
    assert vecinos is not None and len(vecinos) == 3 and i not in list(vecinos), "mas_similares deve restituire 3 indici senza includere la recensione stessa"
    sim = cosine_similarity(X[i], X[list(vecinos)]).ravel()
    assert (np.diff(sim) <= 1e-12).all(), "i vicini devono andare dalla più alla meno simile"
    assert buscar(df_res.loc[i, "texto_limpio"], k=1)[0] == i, "una recensione deve essere la propria vicina più simile quando la cerchi"
consulta_nueva = df_res.loc[3, "texto_limpio"].lower()
assert buscar(consulta_nueva, k=1)[0] == 3, "il testo in minuscolo trova ancora la recensione originale"
print("Sfida finale corretta ✔")

**Le tue conclusioni (scrivi 3-4 frasi):** nella sfida, i vicini di una recensione hanno lo stesso sentimento? Guarda due o tre esempi con `df_res["etiqueta"]`. Cosa cambieresti (n-grammi, `min_df`, `max_df`, caratteri) per migliorare il motore e come misureresti se migliora?

*(La tua risposta qui.)*

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
def bolsa(frases):
    tokens = [re.findall(r"\w+", f.lower()) for f in frases]
    vocabulario = sorted({t for fila in tokens for t in fila})
    matriz = [[fila.count(palabra) for palabra in vocabulario] for fila in tokens]
    return vocabulario, matriz

vocab_es, matriz_es = bolsa(FRASES["spa"])
vocab_it, matriz_it = bolsa(FRASES["ita"])

# Controllo automatico
assert vocab_es is not None and matriz_es is not None and vocab_it is not None and matriz_it is not None, "Completa le righe contrassegnate con TODO"
assert len(vocab_es) == 11 and len(vocab_it) == 12, "il vocabolario ha 11 parole in spagnolo e 12 in italiano"
assert [sum(f) for f in matriz_es] == [4, 4, 6, 4], "la somma di ogni riga è il numero di parole della frase"
assert [sum(f) for f in matriz_it] == [5, 5, 7, 5], "la somma di ogni riga è il numero di parole della frase"
for idioma, vocab, matriz in (("spa", vocab_es, matriz_es), ("ita", vocab_it, matriz_it)):
    cv = CountVectorizer(token_pattern=r"\w+")
    esperada = cv.fit_transform(FRASES[idioma]).toarray()
    assert vocab == list(cv.get_feature_names_out()), "il vocabolario deve essere ordinato e senza ripetizioni"
    assert (np.array(matriz) == esperada).all(), "la matrice non coincide con quella di CountVectorizer"
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
def idf(n, df):
    return float(np.log((1 + n) / (1 + df)) + 1)

tokens = [re.findall(r"\w+", f.lower()) for f in FRASES["spa"]]
n = len(tokens)
df_palabra = Counter(p for fila in tokens for p in set(fila))

idf_entrega = idf(n, df_palabra["entrega"])
pesos_brutos = {p: veces * idf(n, df_palabra[p]) for p, veces in Counter(tokens[0]).items()}
peso_entrega = pesos_brutos["entrega"] / np.sqrt(sum(v ** 2 for v in pesos_brutos.values()))

# Controllo automatico
assert idf_entrega is not None and pesos_brutos is not None and peso_entrega is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(idf(4, 4), 1.0), "una parola presente in tutte le frasi ha idf = 1"
tv = TfidfVectorizer(token_pattern=r"\w+").fit(FRASES["spa"])
assert np.isclose(idf_entrega, tv.idf_[tv.vocabulary_["entrega"]]), "rivedi la formula: ln((1 + n) / (1 + df)) + 1"
assert np.isclose(idf_entrega, np.log(5 / 3) + 1), "entrega compare in 2 delle 4 frasi: ln(5/3) + 1"
bruta = TfidfVectorizer(token_pattern=r"\w+", norm=None).fit(FRASES["spa"])
fila0 = bruta.transform(FRASES["spa"][:1]).toarray()[0]
assert set(pesos_brutos) == set(tokens[0]), "pesos_brutos deve avere una voce per ogni parola distinta della frase 0"
for palabra, peso in pesos_brutos.items():
    assert np.isclose(peso, fila0[bruta.vocabulary_[palabra]]), f"peso errato per {palabra}"
assert np.isclose(peso_entrega, tv.transform(FRASES["spa"][:1])[0, tv.vocabulary_["entrega"]]), "il peso normalizzato è il valore grezzo diviso per la norma L2 della riga"
print(f"Esercizio 2 corretto ✔ (idf di entrega = {idf_entrega:.4f}, peso normalizzato = {peso_entrega:.4f})")

### Soluzione dell'esercizio 3

In [ ]:
def bigramas(frase):
    tokens = re.findall(r"\w+", frase.lower())
    return list(zip(tokens, tokens[1:]))

lista = bigramas(FRASE_NEGACION["spa"])
n_total = len(lista)
n_distintos = len(set(lista))
n_rasgos_1_2 = len(set(re.findall(r"\w+", FRASE_NEGACION["spa"].lower()))) + n_distintos

# Controllo automatico
assert n_total is not None and n_distintos is not None and n_rasgos_1_2 is not None and bigramas("a b") is not None, "Completa le righe contrassegnate con TODO"
assert bigramas("uno dos tres") == [("uno", "dos"), ("dos", "tres")], "un bigramma è una coppia di parole consecutive"
assert n_total == 5 and n_distintos == 4, "la frase ha 5 bigrammi e il primo si ripete"
assert n_rasgos_1_2 == 8, "4 parole distinte + 4 bigrammi distinti"
for idioma, frase in FRASE_NEGACION.items():
    cv_bi = CountVectorizer(token_pattern=r"\w+", ngram_range=(2, 2)).fit([frase])
    assert len(cv_bi.vocabulary_) == len(set(bigramas(frase))) == 4
    cv_12 = CountVectorizer(token_pattern=r"\w+", ngram_range=(1, 2)).fit([frase])
    assert len(cv_12.vocabulary_) == 8
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
u = np.array([2, 1, 0, 1])
v = np.array([1, 1, 1, 0])
w = 2 * u

def coseno(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

cos_uv = coseno(u, v)
cos_uw = coseno(u, w)
cos_vw = coseno(v, w)

# Controllo automatico
assert cos_uv is not None and cos_uw is not None and cos_vw is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(cos_uv, 3 / np.sqrt(18)), "u·v = 3, ‖u‖ = √6, ‖v‖ = √3"
assert np.isclose(cos_uw, 1.0), "w ha la stessa direzione di u"
assert np.isclose(cos_vw, cos_uv), "v si confronta con la stessa direzione, quindi cos(v, w) = cos(v, u)"
esperada = cosine_similarity(np.array([u, v, w]))
assert np.allclose([cos_uv, cos_uw, cos_vw], [esperada[0, 1], esperada[0, 2], esperada[1, 2]]), "deve coincidere con cosine_similarity di scikit-learn"
assert cos_uw > cos_uv, "u è più vicino a w che a v"
print(f"Esercizio 4 corretto ✔ (cos(u, v) = {cos_uv:.4f})")

### Soluzione dell'esercizio 5

In [ ]:
def df_por_palabra(frases):
    return Counter(p for f in frases for p in set(re.findall(r"\w+", f.lower())))

def predecir(frases):
    n = len(frases)
    df = df_por_palabra(frases)
    base = len(df)
    min2 = sum(1 for d in df.values() if d >= 2)
    max075 = sum(1 for d in df.values() if d <= 0.75 * n)
    ambos = sum(1 for d in df.values() if 2 <= d <= 0.75 * n)
    return base, min2, max075, ambos

previsto = {idioma: predecir(FRASES[idioma]) for idioma in ("spa", "ita")}

# Controllo automatico
assert df_por_palabra(["a b", "a c"]) is not None and all(v is not None for t in previsto.values() for v in t), "Completa le righe contrassegnate con TODO"
assert previsto["spa"] == (11, 4, 10, 3), "spagnolo: 11 parole; con min_df=2 restano la, fue, entrega e amable; con max_df=0.75 se ne va fue (è nelle 4 frasi)"
assert previsto["ita"] == (12, 8, 11, 7), "italiano: 12 parole; con max_df=0.75 se ne va è (è nelle 4 frasi)"
for idioma in ("spa", "ita"):
    real = tuple(len(CountVectorizer(token_pattern=r"\w+", **p).fit(FRASES[idioma]).vocabulary_)
                 for p in ({}, {"min_df": 2}, {"max_df": 0.75}, {"min_df": 2, "max_df": 0.75}))
    assert real == previsto[idioma], f"{idioma}: la tua previsione {previsto[idioma]} non coincide con CountVectorizer {real}"
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
IDIOMA = "spa"
df_res = pd.read_csv(cargar_datos(ARCHIVO_RESENAS[IDIOMA]))

vectorizador = TfidfVectorizer(token_pattern=r"\w+")
X = vectorizador.fit_transform(df_res["texto_limpio"])

def mas_similares(i, k=3):
    sim = cosine_similarity(X[i], X).ravel()
    sim[i] = -1
    return list(np.argsort(-sim)[:k])

def buscar(consulta, k=3):
    sim = cosine_similarity(vectorizador.transform([consulta]), X).ravel()
    return list(np.argsort(-sim)[:k])

assert vectorizador is not None and X is not None, "Completa le righe contrassegnate con TODO"
assert sp.issparse(X) and X.shape[0] == len(df_res), "X deve essere una matrice sparsa con una riga per recensione"
assert X.nnz / (X.shape[0] * X.shape[1]) < 0.05, "la matrice dovrebbe avere meno del 5 % di celle non nulle"
assert np.allclose(np.sqrt(X.multiply(X).sum(axis=1)).A1, 1.0), "le righe di TfidfVectorizer hanno norma L2 uguale a 1"
for i in range(0, len(df_res), 25):
    vecinos = mas_similares(i, k=3)
    assert vecinos is not None and len(vecinos) == 3 and i not in list(vecinos), "mas_similares deve restituire 3 indici senza includere la recensione stessa"
    sim = cosine_similarity(X[i], X[list(vecinos)]).ravel()
    assert (np.diff(sim) <= 1e-12).all(), "i vicini devono andare dalla più alla meno simile"
    assert buscar(df_res.loc[i, "texto_limpio"], k=1)[0] == i, "una recensione deve essere la propria vicina più simile quando la cerchi"
consulta_nueva = df_res.loc[3, "texto_limpio"].lower()
assert buscar(consulta_nueva, k=1)[0] == 3, "il testo in minuscolo trova ancora la recensione originale"
print("Sfida finale corretta ✔")

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto della lezione non ti è rimasto chiaro (sacco di parole, n-grammi, TF-IDF di scikit-learn contro quello del libro, dispersione, coseno, `fit` contro `transform`)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*